In [ ]:
# Import necessary packages
import pandas as pd
import plotly.express as px
from scipy.stats import chi2_contingency


In [ ]:
# Read the CSV file 
df = pd.read_csv('df_merged_filled.csv')

# Select and rename relevant columns
df_plot = df[['geo', 'TIME_PERIOD',
              'Consignment_full_train_load_THS_T',
              'Consignment_full_wagon_load_THS_T',
              'Consignment_total_THS_T']].copy()

df_plot.columns = ['Country', 'Year', 'Full_Train', 'Full_Wagon', 'Total']
df_plot['Year'] = df_plot['Year'].astype(int)

#Country name mapping 
country_names = {
    'CH': 'Switzerland',
    'DE': 'Germany',
    'IT': 'Italy',
    'PL': 'Poland',
    'SE': 'Sweden',
    'SI': 'Slovenia',
    'SK': 'Slovakia'
}
df_plot['Country_Name'] = df_plot['Country'].map(country_names)

#Plot 1: Full Train Load
fig1 = px.line(
    df_plot,
    x='Year',
    y='Full_Train',
    color='Country_Name',
    markers=True,
    title='Full Train Load Consignments by Country (Thousand Tonnes)',
    labels={'Full_Train': 'Thousand Tonnes', 'Country_Name': 'Country'}
)
fig1.update_layout(title_x=0.5, legend_title_text='Country', hovermode='x unified')
fig1.show()

#Plot 2: Full Wagon Load
fig2 = px.line(
    df_plot,
    x='Year',
    y='Full_Wagon',
    color='Country_Name',
    markers=True,
    title='Full Wagon Load Consignments by Country (Thousand Tonnes)',
    labels={'Full_Wagon': 'Thousand Tonnes', 'Country_Name': 'Country'}
)
fig2.update_layout(title_x=0.5, legend_title_text='Country', hovermode='x unified')
fig2.show()

#Plot 3: Total Consignments
fig3 = px.line(
    df_plot,
    x='Year',
    y='Total',
    color='Country_Name',
    markers=True,
    title='Total Freight Consignments by Country (Thousand Tonnes)',
    labels={'Total': 'Thousand Tonnes', 'Country_Name': 'Country'}
)
fig3.update_layout(title_x=0.5, legend_title_text='Country', hovermode='x unified')
fig3.show()

#Prepare the data for chi-square test
# Aggregate the total consignment types per country
contingency_table = df.groupby('geo')[[
    'Consignment_full_train_load_THS_T',
    'Consignment_full_wagon_load_THS_T'
]].sum()

# Optional: rename columns for readability
contingency_table.columns = ['Full Train Load', 'Full Wagon Load']

# Perform the Chi-square test of independence
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

# --- Print the results ---
print("Chi-square Statistic:", chi2)
print("Degrees of Freedom:", dof)
print("P-value:", p_value)
print("\nExpected Frequencies:\n", pd.DataFrame(expected, 
      index=contingency_table.index, 
      columns=contingency_table.columns))

# --- Interpretation ---
alpha = 0.05  # significance level
if p_value < alpha:
    print("\n Reject H0: There IS a significant relationship between country and consignment type distribution.")
else:
    print("\n Fail to Reject H0: There is NO significant relationship between country and consignment type distribution.")
